# 상품 주문으로 설명하는 DB 발표

**실행:** 로그인 → 상품 확인/등록 → 주문 → 주문상품 확인 → 배송중으로 수정.

회원 계정과 해당 `user_details` 행은 미리 준비한다. **위에서부터 코드 셀 실행 → 바로 아래 설명 읽기** 순서로 진행한다. 각 코드 셀은 이전 셀에서 만든 변수를 이어 사용한다. 생성/수정된 행은 Supabase Table Editor에서 직접 확인한다. 셀을 다시 실행하면 등록/주문 요청도 다시 실행된다.

기존 클래스는 그대로 사용한다. `ex`에는 새 import, 새 클래스, 예외 처리를 넣지 않는다. 배송중은 상품 원본의 상태가 아니라 **해당 주문의 상태**이므로 `orders.order_status`를 수정한다.

발표 범위는 아래 1~6이다. 보류한 DDL/DML 설명과 7번 주제는 제외한다.

In [ ]:
%run ./orders.ipynb
%run ./order_items.ipynb

order_service = Order_service(supabase)
order_item_service = OrderItemService(supabase)

## 준비. 기존 노트북 불러오기

기존 파일에 있는 연결 코드와 클래스를 사용한다. `products.ipynb` 전체에는 별도의 로그인 실습이 있으므로 실행하지 않고, 상품 조회/등록만 여기서 짧게 작성한다.

In [ ]:
email = input("실습 계정 이메일: ")
password = input("실습 계정 비밀번호: ")

login = supabase.auth.sign_in_with_password({
    "email": email, "password": password
})

user_id = login.user.id
password = None

print("로그인한 회원 ID:", user_id)

## 시연 A. 로그인

GUI의 로그인 폼에 해당한다. 로그인 결과의 사용자 ID를 주문자 ID로 사용한다. 입력한 비밀번호와 로그인 응답 전체는 출력하지 않는다.

In [ ]:
product_name = "발표용 머그컵"
products = (
    supabase.table("products")
            .select("*")
            .eq("name", product_name)
            .is_("deleted_at", "null")
            .execute().data
)

print("상품 있음" if products else "상품 없음: 새로 등록합니다.")

if not products:
    products = supabase.table("products").insert({
        "name": product_name, "price": 12000,
        "description": "발표용 상품", "seller_id": user_id,
    }).execute().data

product = products[0]
print(product)

## 시연 B. 상품이 있는지 확인하고 없으면 등록

상품명으로 조회하고 결과 리스트가 비었으면 실습용 상품 한 건을 만든다. `seller_id`는 `user_details.id`를 참조하므로 앞에서 준비한 회원 ID를 넣는다. 등록 후 `products` 테이블을 직접 확인한다.

In [ ]:
quantity = 2

orders = order_service.create_order({
    "order_no": f"EX-{uuid.uuid4().hex}",
    "order_name": "발표용 주문자",
    "user_id": user_id,
    "total_price": product["price"] * quantity,
    "order_status": "ORDER",
})

order_id = orders[0]["id"]

items = order_item_service.create_order_item(
    order_id, product["id"], product["name"], product["price"], quantity
)

print("주문 ID:", order_id)
print(items)

## 시연 C. 상품 주문

먼저 `orders`를 만들고, 반환된 주문 ID와 상품 ID를 `order_items`에 넣는다. 이 셀 한 번이 주문하기 버튼이다. `order_no`와 `order_name`은 실제 테이블의 필수값이며, 주문번호에는 기존 노트북에서 불러온 `uuid`를 사용한다.

**Supabase 확인:** `orders.id = order_items.order_id`, `products.id = order_items.product_id`가 각각 같은지 보여준다. 상품 원본이 하나 더 생기는 것이 아니라 **주문상품 행**이 생긴다.

In [ ]:
ordered_items = (supabase.table("order_items")
                        .select("*")
                        .eq("order_id", order_id)
                        .eq("product_id", product["id"])
                        .is_("deleted_at", "null").execute().data
)

print("주문상품 있음" if ordered_items else "주문상품 없음")
print(ordered_items)

## 시연 D. 주문에 해당 상품이 들어갔는지 확인

주문 ID와 상품 ID로 조회한다. 결과가 있으면 이 주문에 해당 상품이 연결된 것이다.

In [ ]:
updated_order = order_service.update_order_status(user_id, order_id, "ON_DELIVERY")

print(updated_order)

## 시연 E. 배송중으로 수정

배송이 시작됐다고 가정하고 기존 메서드로 주문 상태를 바꾼다. Supabase의 `orders`에서 `order_status`가 **주문접수 → 배송중**으로 바뀌는 것을 보여준다.

## 1. 테이블 관계 표현

담당: 이승주, 배서인, 서종찬, 양석호

- `auth.users` ↔ `user_details`: 계정 하나에 상세 정보가 최대 한 행인 1:1 관계.
- 회원 → `orders`: 한 회원이 여러 번 주문하는 1:N 관계.
- `orders` → `order_items`: 주문 하나에 여러 상품 항목이 들어가는 1:N 관계.
- `products` → `order_items`: 같은 상품이 여러 주문에 들어가는 1:N 관계.
- 결과적으로 주문과 상품의 N:M 관계를 `order_items`가 연결한다.

**발표:** “FK는 연결할 행이 존재하는지 확인합니다. 주문상품을 자동 생성하는 것은 FK가 아니라 방금 실행한 Python 코드입니다.”

## 2. 데이터 집계함수

담당: 오명석, 양석호

아래 SQL은 Python 셀이 아니라 **Supabase SQL Editor에서 실행할 조회 예제**다. 주문 접수/배송중 등 상태별 주문 수와 금액을 묶어 본다.

```sql
SELECT order_status,
       COUNT(*) AS order_count,
       SUM(total_price) AS total_amount,
       AVG(total_price) AS average_amount,
       MIN(total_price) AS minimum_amount,
       MAX(total_price) AS maximum_amount
FROM orders
WHERE deleted_at IS NULL
GROUP BY order_status;
```

**발표:** “COUNT는 건수, SUM은 합계, AVG는 평균, MIN/MAX는 최솟값/최댓값입니다. GROUP BY로 배송 상태별 결과를 볼 수 있습니다.”

## 3. 집합과 조인

담당: 장예지, 양석호

**JOIN**은 관계 있는 테이블의 컬럼을 옆으로 연결한다. 아래는 주문, 주문상품, 상품을 연결해 읽는 예제다. INNER JOIN은 연결되는 행만, LEFT JOIN은 왼쪽 행을 유지하고 연결이 없으면 오른쪽 값을 NULL로 표시한다.

```sql
SELECT o.order_no, p.name, oi.quantity, o.order_status
FROM orders o
JOIN order_items oi ON oi.order_id = o.id
JOIN products p ON p.id = oi.product_id
WHERE o.deleted_at IS NULL AND oi.deleted_at IS NULL;
```

**집합 연산**은 조회 결과의 행을 합치거나 비교한다. UNION은 중복 제거 합집합, UNION ALL은 중복 유지, INTERSECT는 교집합, EXCEPT는 차집합이다. 합칠 결과의 컬럼 개수와 자료형이 맞아야 한다.

```sql
SELECT user_id FROM orders WHERE order_status = '주문접수'
UNION
SELECT user_id FROM orders WHERE order_status = '배송중';
```

**발표:** “JOIN으로 주문에 상품명을 붙이고, UNION으로 주문접수 또는 배송중인 주문자의 목록을 합칠 수 있습니다.”

## 4. 서브쿼리 유형

담당: 이승주, 장예지, 양석호

서브쿼리는 다른 SQL 안에 들어가는 조회다. **반환 결과 기준**으로는 단일 값/단일 행/다중 행을 구분하고, **바깥 쿼리 참조 여부**로는 비상관/상관 서브쿼리를 구분한다.

```sql
-- 단일 값을 반환하는 비상관 서브쿼리: 평균보다 비싼 상품
SELECT name, price FROM products
WHERE price > (SELECT AVG(price) FROM products);

-- 여러 행을 반환하는 서브쿼리: 주문된 적 있는 상품
SELECT name FROM products
WHERE id IN (SELECT product_id FROM order_items);

-- 상관 서브쿼리: 바깥 상품 p.id를 참조해 주문 존재 확인
SELECT p.name FROM products p
WHERE EXISTS (
    SELECT 1 FROM order_items oi WHERE oi.product_id = p.id
);
```

위 예제는 삭제 이력까지 포함한 전체 데이터를 기준으로 한다. 위치로 분류하면 SELECT 안의 스칼라 서브쿼리, FROM 안의 인라인 뷰, WHERE 안의 조건용 서브쿼리도 있다.

**발표:** “평균값을 구해서 비교할 수도 있고, 다른 조회 결과에 포함되는지 IN으로 확인하거나 관련 행의 존재를 EXISTS로 확인할 수도 있습니다.”

## 5. 트랜잭션(TCL)과 ACID 원칙

담당: 서종찬, 배서인, 장예지, 양석호

트랜잭션은 여러 작업을 하나의 처리 단위로 묶는다. TCL의 COMMIT은 확정, ROLLBACK은 취소, SAVEPOINT는 중간 지점을 만든다. PostgreSQL에서는 BEGIN으로 트랜잭션을 시작한다.

```sql
BEGIN;
-- 아래 ID를 시연에서 출력한 실제 주문 ID로 바꾼다.
UPDATE orders SET order_status = '배송완료'
WHERE id = '여기에-주문-UUID';
ROLLBACK; -- 변경을 취소하므로 원래 배송중 상태가 유지된다.
-- 확정하려면 ROLLBACK 대신 COMMIT을 사용한다.
```

- **A, 원자성:** 주문과 주문상품이 모두 저장되거나 모두 취소되어야 한다.
- **C, 일관성:** FK, 수량 같은 무결성 조건을 지키며 처리한다.
- **I, 격리성:** 동시 작업이 서로에게 보이는 방식을 격리 수준에 따라 제어한다.
- **D, 지속성:** 확정된 결과는 장애 후에도 유지되어야 한다.

**발표:** “지금 ex의 두 저장 호출은 각각 별도 API 요청이므로 한 트랜잭션은 아닙니다. 두 요청을 원자적으로 처리하려면 DB 함수 등의 서버 처리로 묶어야 합니다. 이번에는 개념까지만 설명합니다.”

## 6. SDK, 랭체인

담당: 배서인, 장예지, 오명석, 양석호

**SDK**는 서비스를 코드에서 사용하도록 제공하는 개발 도구다. 지금의 `supabase.table(...).select(...).execute()`가 Supabase Python SDK 사용 예다. SDK가 API 요청을 구성하고 서버가 DB 작업을 수행한다.

**LangChain**은 LLM과 도구, 검색 등의 호출을 연결하는 프레임워크다. 예를 들어 “내 주문이 배송중인지 알려줘”라는 질문을 주문 조회 도구와 연결할 수 있다. 이번 예제에는 LangChain을 설치하거나 호출하지 않는다.

**발표:** “Supabase SDK로 DB 기능을 호출했고, 이 기능을 자연어 질문과 연결하는 확장에는 LangChain 같은 도구를 사용할 수 있습니다. 현재 주문 예제는 SDK만 사용합니다.”

---
[같은 내용의 Notion 발표 자료](https://app.notion.com/p/3d8dc8b0e1638100ab71d26bd7b39870)

발표는 **시연 A~E → 관계 → 집계 → 집합/JOIN → 서브쿼리 → 트랜잭션 → SDK/LangChain** 순서로 진행한다. 실제 DB 실행은 발표자가 수행한다.

In [ ]:
admin_orders = (supabase.table("orders")
    .select("id,order_no,total_price,order_status")
    .is_("deleted_at", "null").execute().data)
admin_products = (supabase.table("products")
    .select("id,name,price").is_("deleted_at", "null").execute().data)
admin_items = (supabase.table("order_items")
    .select("order_id,product_id,quantity,item_price")
    .is_("deleted_at", "null").execute().data)

print("[관리자 조회] 주문", len(admin_orders), "건 / 상품", len(admin_products), "개")
for row in admin_orders:
    print(row["order_no"], row["order_status"], row["total_price"])

In [ ]:
print(f"{'상태':<16} {'COUNT':>6} {'SUM':>10} {'AVG':>10} {'MIN':>10} {'MAX':>10}")
for status in sorted({row["order_status"] or "미지정" for row in admin_orders}):
    rows = [row for row in admin_orders if (row["order_status"] or "미지정") == status]
    amounts = [row["total_price"] for row in rows if row["total_price"] is not None]
    total = sum(amounts) if amounts else None
    average = round(total / len(amounts), 2) if amounts else None
    print(f"{status:<16} {len(rows):>6} {str(total):>10} {str(average):>10} "
          f"{str(min(amounts, default=None)):>10} {str(max(amounts, default=None)):>10}")

In [ ]:
joined_orders = (supabase.table("orders")
    .select("order_no,order_status,order_items(item_name,item_price,quantity,products(name))")
    .is_("deleted_at", "null")
    .is_("order_items.deleted_at", "null")
    .execute().data)

for order in joined_orders:
    for item in order["order_items"]:
        current_product = item["products"]
        current_name = current_product["name"] if current_product else "연결 상품 없음"
        print(order["order_no"], order["order_status"], current_name,
              "수량:", item["quantity"], "주문금액:", item["item_price"] * item["quantity"])

In [ ]:
registered_ids = {row["id"] for row in admin_products}
ordered_ids = {row["product_id"] for row in admin_items}

print("UNION 합집합:", registered_ids | ordered_ids)
print("INTERSECT 교집합, 등록돼 있고 주문된 상품:", registered_ids & ordered_ids)
print("EXCEPT 차집합, 등록돼 있지만 주문되지 않은 상품:", registered_ids - ordered_ids)
print("UNION ALL, 중복 유지:",
      [row["id"] for row in admin_products] + [row["product_id"] for row in admin_items])

In [ ]:
prices = [row["price"] for row in admin_products if row["price"] is not None]
average_price = sum(prices) / len(prices) if prices else None
expensive_products = [row for row in admin_products
    if row["price"] is not None and average_price is not None and row["price"] > average_price]
print("전체 상품 평균 가격:", average_price)
print("평균보다 비싼 상품:", expensive_products)

In [ ]:
ordered_products = [row for row in admin_products if row["id"] in ordered_ids]
print("주문상품 기록에 등장하는 상품:")
for row in ordered_products:
    print(row["name"], row["price"])

In [ ]:
for row in admin_products:
    has_order = any(item["product_id"] == row["id"] for item in admin_items)
    print(row["name"], "주문 기록 있음" if has_order else "주문 기록 없음")

In [ ]:
transaction_sql = f"""BEGIN;
UPDATE orders SET order_status = 'DELIVERED' WHERE id = '{order_id}';
SELECT id, order_status FROM orders WHERE id = '{order_id}';
ROLLBACK;
SELECT id, order_status FROM orders WHERE id = '{order_id}';"""
print(transaction_sql)

In [ ]:
query = supabase.table("orders").select("id,order_status")
same_query = query.eq("id", order_id)

print("eq()가 원래 객체를 반환했나요?", query is same_query)
print(same_query.is_("deleted_at", "null").execute().data)